# Лабораторная работа №7

Сложение векторов в CUDA.

Напишите CUDA-программу, которая вычисляет сумму двух векторов
размера n. Используйте следующие формулы:
𝐴𝐴 = (𝑎𝑎1 , 𝑎𝑎2 , … , 𝑎𝑎𝑛𝑛 )
𝐵𝐵 = (𝑏𝑏1 , 𝑏𝑏2 , … , 𝑏𝑏𝑛𝑛 )
𝐶𝐶 = (𝑎𝑎1 + 𝑏𝑏1 , 𝑎𝑎2 + 𝑏𝑏2 , … , 𝑎𝑎𝑛𝑛 + 𝑏𝑏𝑛𝑛 )

Входные данные: целое число 𝑛𝑛 ≥ 1, 𝑛𝑛 целых элементов вектора 𝐴𝐴 и 𝑛𝑛
целых элементов вектора 𝐵𝐵.

Выходные данные: 𝑛𝑛 целых элементов вектора 𝐶𝐶.

In [4]:
%%writefile vector_add.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void vectorAdd(int* A, int* B, int* C, int n)
{
    int threadId = blockIdx.x * blockDim.x + threadIdx.x;
    if (threadId < n)
    {
        C[threadId] = A[threadId] + B[threadId];
    }
}

int main()
{
    int n;
    std::cin >> n;

    // Выделение памяти на хосте
    int* hA = new int[n];
    int* hB = new int[n];
    int* hC = new int[n];

    // Циклы считывания данных с клавиатуры в массивы на CPU
    for (int i = 0; i < n; i++) hA[i] = random();
    for (int i = 0; i < n; i++) hB[i] = random();

    // Выделение памяти на устройстве (GPU)
    int* dA, *dB, *dC;
    cudaMalloc((void**)&dA, n * sizeof(int));
    cudaMalloc((void**)&dB, n * sizeof(int));
    cudaMalloc((void**)&dC, n * sizeof(int));

    // Копирование данных с хоста на GPU
    cudaMemcpy(dA, hA, n * sizeof(int), cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, n * sizeof(int), cudaMemcpyHostToDevice);

    // Вычисление размера грида
    int blockSize = 32;
    int gridSize = (n - 1) / blockSize + 1;

    // Запуск ядра
    vectorAdd<<<gridSize, blockSize>>>(dA, dB, dC, n);

    // Копирование результата с GPU на хост
    cudaMemcpy(hC, dC, n * sizeof(int), cudaMemcpyDeviceToHost);

    // Вывод результата
    for (int i = 0; i < n; i++)
    {
        std::cout << hC[i];
        if (i < n - 1) std::cout << " ";
    }
    std::cout << std::endl;

    // Освобождение памяти
    cudaFree(dA);
    cudaFree(dB);
    cudaFree(dC);
    delete[] hA;
    delete[] hB;
    delete[] hC;

    return 0;
}

Overwriting vector_add.cu


In [8]:
!nvcc vector_add.cu -o vector_add && echo "127464" | ./vector_add

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
-1360952787 -1736106698 -1192350032 1919045306 -1048599574 -1743829828 1562441389 -787709848 1284607342 1804499641 1312120875 -1885909347 -1408810172 -1324063484 -2125573920 1999484397 1679558970 -1538186414 1328123690 1708936545 1431635810 918773299 1718150908 1990634251 486973262 1839041830 898357125 1273938258 -1469444053 1711836299 -1106254571 -682913192 2123213251 1996362693 -911351534 -1072869971 -1894950782 -1496393791 -1860579819 -610343440 -1839377798 -548458944 1798714509 -1100704322 -1872522428 1820624239 -1248703571 -192963458 282437825 -2068063529 1515973088 1714073635 998193420 -1060843300 -590259408 1485166682 778198531 308097717 -1535862356 1456238126 2019934016 -494633278 -1374158713 1995663619 1501729416 2009457049 922793648 1754262282 -1634420390 -937786171 1143918844 821169110 -1486

### 1. Расскажите о функциях управления памятью на графическом ускорителе.

В CUDA память GPU и CPU физически разделены, поэтому для работы с данными на GPU используются специальные функции.

cudaMalloc(void** devPtr, size_t size) - выделяет память на GPU. Принимает указатель на указатель и размер в байтах. Память не обнуляется автоматически.

Адрес выровнен по 512 байт для оптимального доступа.
cudaFree(void* devPtr) - освобождает ранее выделенную память на GPU.

Обязательно вызывать после завершения работы, иначе будет утечка памяти устройства.
cudaMemcpy(void* dst, const void* src, size_t count, cudaMemcpyKind kind) - копирует данные между CPU и GPU. Направление задаётся параметром kind:

- cudaMemcpyHostToDevice - с CPU на GPU (загрузка входных данных)
- cudaMemcpyDeviceToHost - с GPU на CPU (получение результата)
- cudaMemcpyHostToHost - между двумя областями на CPU
- cudaMemcpyDeviceToDevice - между двумя областями на GPU

В нашей программе это выглядело так: сначала загружаем векторы A и B с CPU на GPU через HostToDevice, запускаем ядро, затем забираем результат C обратно через DeviceToHost.

### 2. Какой размер грида для данной задачи даст наибольшее ускорение решения?

Наибольшее ускорение достигается когда количество нитей максимально соответствует количеству элементов вектора, то есть один элемент - одна нить. При этом каждый элемент вычисляется полностью параллельно.

Оптимальный размер блока - 32 нити (или кратное 32: 64, 128, 256), поскольку GPU выполняет нити группами по 32 штуки, которые называются warp. Если размер блока не кратен 32, часть нитей последнего warp простаивает впустую.

Для нашей задачи при n = 4 и blockSize = 32 грид будет состоять из 1 блока, что уже оптимально. При больших n, например 1024, будет 32 блока по 32 нити - все элементы считаются за один параллельный шаг.